<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [17]:
import json, math, re, unicodedata
from collections import Counter
import spacy

try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)               # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [1]:
# ! pip install fasttext
import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    return ft.get_word_vector(palabra)

print("Dimensión del embedding:", ft.get_dimension())
print("Tamaño del vocabulario:", len(ft.get_words()))

Dimensión del embedding: 300
Tamaño del vocabulario: 2000000


**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [3]:
# TODO: para 'sequia', 'cafe', 'chiapas', imprimir sus 5 vecinos mas cercanos
#       (ft.get_nearest_neighbors). Comenten cualquier vecino sorprendente.

for palabra in ['sequia', 'cafe', 'chiapas']:
    print(f"\n Vecinos de '{palabra}':")
    for score, vecino in ft.get_nearest_neighbors(palabra, k=5):
        print(f"  {score:.4f}  {vecino}")


→ Vecinos de 'sequia':
  0.7464  sequía
  0.7236  sequias
  0.5896  inundacion
  0.5854  escacez
  0.5713  sequías

→ Vecinos de 'cafe':
  0.7897  café
  0.7414  cafes
  0.7375  cafe.
  0.7242  cafe-
  0.7142  cafesito

→ Vecinos de 'chiapas':
  0.7302  chiapas.
  0.7262  oaxaca
  0.7059  tuxtla
  0.6912  michoacan
  0.6861  veracruz


####Vecinos interesantes:
- Sequías -> Inundación

**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [5]:
import numpy as np
def cos_vec(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    raise NotImplementedError

# TODO: comparar coseno(agua, hidrico) (esperado ALTO) vs coseno(agua, jirafa) (esperado BAJO)
sim_alta = cos_vec(vec('agua'), vec('hidrico'))
sim_baja = cos_vec(vec('agua'), vec('jirafa'))

print(f"coseno(agua, hidrico) = {sim_alta:.4f}")
print(f"coseno(agua, jirafa)  = {sim_baja:.4f}")

coseno(agua, hidrico) = 0.4360
coseno(agua, jirafa)  = 0.2433


_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_

**A.4** Analogías por aritmética vectorial.

In [11]:
# TODO: probar rey - hombre + mujer (ft.get_analogies) y una analogia de capitales.
#       Documenten un caso que funcione y uno que falle.

print("rey - hombre + mujer:")
for score, palabra in ft.get_analogies("rey", "hombre", "mujer", k=5):
    print(f"  {score:.4f}  {palabra}")

print("\nFrancia - París + Roma:")
for score, palabra in ft.get_analogies("Francia", "París", "Roma", k=5):
    print(f"  {score:.4f}  {palabra}")

print("\nPlaya - verano + invierno:")  # relación estacional abstracta
for score, p in ft.get_analogies("playa", "verano", "invierno", k=5):
    print(f"  {score:.4f}  {p}")

rey - hombre + mujer:
  0.6996  reina
  0.6584  princesa
  0.5786  reina-madre
  0.5746  monarca
  0.5572  emperatriz

Francia - París + Roma:
  0.7544  Italia
  0.5961  Grecia
  0.5961  Roma.A
  0.5684  Roma.La
  0.5582  Roma.En

Playa - verano + invierno:
  0.6612  playa.A
  0.6604  playa.La
  0.6560  playa.Y
  0.6384  laplaya
  0.6217  playa.En


:_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_

Falla en la tercera opcion, ya que es un caso donde la relación es basatante abstracta o no hay una equivalencia tan marcada

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [15]:
def vector_documento(tokens):
    if not tokens:
        return np.zeros(ft.get_dimension())
    vectores = np.array([vec(token) for token in tokens])
    return vectores.mean(axis=0)


# TODO: construir EMB_DOCS = {id: vector_documento(tokens)} para todo el corpus
EMB_DOCS = {doc_id: vector_documento(tokens)
            for doc_id, tokens in zip(ids, documentos)}

print(f"Documentos embebidos: {len(EMB_DOCS)}")
print(f"Dimensión de cada vector: {next(iter(EMB_DOCS.values())).shape}")

Documentos embebidos: 14
Dimensión de cada vector: (300,)


**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

In [18]:
# TODO: peguen su preprocesar() del Lab 1.

RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_EMOJI = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]')

stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'sin'}   # palabras que NO quieren tratar como stopword (con criterio de dominio)
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto, quitar_acentos=True):
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.lower()
    texto = RE_HTML.sub(' ', texto)
    texto = RE_URL.sub(' ', texto)
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower()
        if lemma in MIS_STOPWORDS:
            continue
        if len(lemma) <= 1 and not lemma.isdigit():
            continue
        tokens.append(lemma)
    return tokens


def buscar_semantico(consulta, k=5):
    # TODO: preprocesar la consulta -> vector_documento -> coseno contra EMB_DOCS -> top-k
    tokens  = preprocesar(consulta)
    v_query = vector_documento(tokens)

    scores = {
        doc_id: cos_vec(v_query, v_doc)
        for doc_id, v_doc in EMB_DOCS.items()
    }

    top_k = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

    print(f"\nConsulta: '{consulta}'  →  tokens: {tokens}\n")
    for doc_id, score in top_k:
        print(f"  {score:.4f}  {doc_id}  {titulos[doc_id]}")

# prueba: buscar_semantico('problemas de agua') deberia recuperar d02 (crisis hidrica)
buscar_semantico('problemas de agua')


Consulta: 'problemas de agua'  →  tokens: ['problema', 'agua']

  0.7686  d13  Restablecen servicio de agua potable
  0.6681  d02  Crisis hidrica golpea la region
  0.5604  d11  Alertan por casos de dengue
  0.5230  d04  Sequia afecta cultivos de maiz
  0.5093  d01  Lluvias provocan inundaciones en Tuxtla


**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [22]:
def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    total = len(doc)
    if total == 0:
        return {}
    counts = Counter(doc)
    return {term: count / total for term, count in counts.items()}

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf."""
    n_docs = len(corpus)
    df = Counter()
    for doc in corpus:
        for term in set(doc):  # set() asegura contar el doc una sola vez por término
            df[term] += 1
    return {term: math.log(n_docs / df_t) for term, df_t in df.items()}

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento."""
    tf_doc = tf(doc)
    return {term: tf_val * idf_.get(term, 0) for term, tf_val in tf_doc.items()}
def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF del corpus."""
    texto_proc = preprocesar(texto)
    return tfidf(texto_proc, IDF)

def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    # Producto punto: solo recorremos las claves de v1 (las que no estan en v2 aportan 0)
    dot = sum(peso * v2.get(term, 0) for term, peso in v1.items())

    norma1 = math.sqrt(sum(peso ** 2 for peso in v1.values()))
    norma2 = math.sqrt(sum(peso ** 2 for peso in v2.values()))

    if norma1 == 0 or norma2 == 0:
        return 0.0

    return dot / (norma1 * norma2)

def buscar(consulta, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(consulta)

    resultados = [
        (doc['id'], doc['titulo'], coseno(q, vector))
        for doc, vector in zip(corpus, INDICE)
    ]

    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados[:k]


In [25]:
avgdl = sum(len(doc) for doc in documentos) / len(documentos)

def idf_bm25(corpus):
    # TODO: ln(1 + (N - df + 0.5)/(df + 0.5))  por termino
    n_docs = len(corpus)
    df = Counter()
    for doc in corpus:
        for term in set(doc):
            df[term] += 1
    return {
        term: math.log(1 + (n_docs - df_t + 0.5) / (df_t + 0.5))
        for term, df_t in df.items()
    }


def bm25(doc, q, k1=1.5, b=0.75):
    # TODO: acumular el score termino por termino segun la formula
    counts = Counter(doc)
    dl = len(doc)
    score = 0.0
    for term in q:
        f = counts.get(term, 0)
        if f == 0:
            continue
        idf_t = IDF_BM25.get(term, 0)
        numerador = f * (k1 + 1)
        denominador = f + k1 * (1 - b + b * (dl / avgdl))
        score += idf_t * (numerador / denominador)
    return score

# Construir IDF de BM25 sobre todo el corpus (una sola vez)
IDF_BM25 = idf_bm25(documentos)

def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    # TODO: preprocesar la consulta -> score bm25 contra cada doc -> top-k
    q = preprocesar(consulta)
    resultados = [
        (doc['id'], doc['titulo'], bm25(d, q, k1=k1, b=b))
        for doc, d in zip(corpus, documentos)
    ]
    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados[:k]

IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

In [30]:
# TODO: para 3 consultas, imprimir el top-5 de TF-IDF (Lab 2), BM25 (Lab 3) y semantico, lado a lado.
#       Marquen en cuales el semantico encuentra documentos que los otros dos no.
consultas = [
    'problemas de agua',
    'cafe y cacao chiapas',
    'tecnologia e inteligencia artificial'
]

def comparar_motores(consulta, k=5):
    res_tfidf = buscar(consulta, k=k)
    res_bm25  = buscar_bm25(consulta, k=k)

    tokens  = preprocesar(consulta)
    v_query = vector_documento(tokens)
    scores  = sorted(EMB_DOCS.items(), key=lambda x: cos_vec(v_query, x[1]), reverse=True)[:k]
    res_sem = [(doc_id, titulos[doc_id], float(cos_vec(v_query, v_doc)))   # ← float()
               for doc_id, v_doc in scores]

    ids_tfidf  = {r[0] for r in res_tfidf}
    ids_bm25   = {r[0] for r in res_bm25}
    ids_sem    = {r[0] for r in res_sem}
    exclusivos = ids_sem - ids_tfidf - ids_bm25

    print(f"\n{'='*65}")
    print(f"Consulta: '{consulta}'")
    print(f"{'='*65}")
    print(f"{'#':<3} {'TF-IDF':<22} {'BM25':<22} {'Semántico':<22}")
    print(f"{'-'*65}")
    for i in range(k):
        def fmt(res):
            doc_id, titulo, score = res[i]
            marca = " (New)" if doc_id in exclusivos else ""
            return f"{doc_id}({score:.2f}){marca}"
        print(f"{i+1:<3} {fmt(res_tfidf):<22} {fmt(res_bm25):<22} {fmt(res_sem):<22}")

    if exclusivos:
        print(f"\nExclusivos del semántico: {exclusivos}")
        for doc_id in exclusivos:
            print(f"   {doc_id}: {titulos[doc_id]}")
    else:
        print("\n(Sin exclusivos semánticos en esta consulta)")

for consulta in consultas:
    comparar_motores(consulta)

for consulta in consultas:
    comparar_motores(consulta)


Consulta: 'problemas de agua'
#   TF-IDF                 BM25                   Semántico             
-----------------------------------------------------------------
1   d13(0.48)              d13(3.25)              d13(0.77)             
2   d01(0.00)              d01(0.00)              d02(0.67)             
3   d02(0.00)              d02(0.00)              d11(0.56) (New)       
4   d03(0.00)              d03(0.00)              d04(0.52)             
5   d04(0.00)              d04(0.00)              d01(0.51)             

Exclusivos del semántico: {'d11'}
   d11: Alertan por casos de dengue

Consulta: 'cafe y cacao chiapas'
#   TF-IDF                 BM25                   Semántico             
-----------------------------------------------------------------
1   d12(0.25)              d12(3.63)              d12(0.64)             
2   d03(0.18)              d03(3.03)              d08(0.62)             
3   d08(0.13)              d08(1.81)              d03(0.61)             
4 

**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?

In [34]:
# TODO: reusen sus qrels y su nDCG del Lab 3; calculen nDCG medio para TF-IDF, BM25 y semantico.
#       ¿En que consultas mejora mas el semantico?

qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},   # ejemplo dado

    'precio del cafe': {
        'd03': 3,  # tokens incluyen 'cafe', 'precio', 'alza', 'exportacion' -> tema central
        'd12': 2,  # 'cafe', 'cacao', 'venta', 'productor' -> mismo sector pero el foco es la feria, no el precio
        'd08': 1,  # 'produccion', 'cacao', 'mercado', 'premium' -> mismo rubro agroexportador, pero ni cafe ni precio
    },

    'turismo en chiapas': {
        'd05': 3,  # 'visitante', 'recorrido', 'atractivo', 'parque nacional' -> articulo central de turismo
        'd09': 3,  # 'destino', 'cultural', 'atraer', 'viajero' -> igualmente articulo central de turismo
        'd12': 1,  # 'feria', 'asistente', 'recorrer', 'stand' -> evento que atrae publico, pero no es nota de turismo
    },

    'escasez de agua': {
        'd02': 3,  # 'crisis', 'hidrico', 'desabasto', 'escasez' -> tema central exacto
        'd13': 2,  # 'agua', 'potable', 'falla', 'red' -> problema de suministro de agua, causa distinta (infraestructura, no sequia)
        'd04': 1,  # 'sequia' aparece, pero el foco del articulo es perdida de cultivos, no el agua en si
    },

    'tecnologia e innovacion en chiapas': {
        'd07': 3,  # 'inteligencia', 'artificial', 'laboratorio', 'aprendizaje', 'automatico' -> tema central
        'd14': 3,  # 'robotica', 'robotico', 'ingenieria', 'competencia', 'internacional' -> tema central
        'd10': 1,  # 'obra', 'carretera', 'mejorar' -> es infraestructura, no tecnologia, pero comparte el espiritu de "modernizacion"
    },
}
assert len(qrels) >= 5, 'Faltan consultas por etiquetar'

def _rel(qid, doc): return qrels[qid].get(doc, 0)
def _relevantes(qid): return {d for d, g in qrels[qid].items() if g > 0}


def ndcg_at_k(ranking, qid, k=5):
    def dcg(relevancias):
        # i empieza en 0 -> log2(i+2): la primera posicion usa log2(2)=1
        return sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevancias))

    top_k = ranking[:k]
    rels_obtenidos = [_rel(qid, doc_id) for doc_id, _, _ in top_k]
    dcg_val = dcg(rels_obtenidos)

    rels_ideales = sorted(qrels[qid].values(), reverse=True)[:k]
    idcg_val = dcg(rels_ideales)

    return dcg_val / idcg_val if idcg_val > 0 else 0.0


def ranking_semantico(consulta, k=5):
    """Devuelve lista (doc_id, titulo, score) igual que buscar() y buscar_bm25()."""
    tokens  = preprocesar(consulta)
    v_query = vector_documento(tokens)
    scores  = sorted(EMB_DOCS.items(), key=lambda x: float(cos_vec(v_query, x[1])), reverse=True)[:k]
    return [(doc_id, titulos[doc_id], float(cos_vec(v_query, EMB_DOCS[doc_id]))) for doc_id, _ in scores]

# Calcular nDCG por consulta y por motor
print(f"\n{'Consulta':<35} {'TF-IDF':>8} {'BM25':>8} {'Semántico':>10}")
print('-' * 65)

ndcgs = {'tfidf': [], 'bm25': [], 'sem': []}

for qid in qrels:
    r_tfidf = buscar(qid, k=5)
    r_bm25  = buscar_bm25(qid, k=5)
    r_sem   = ranking_semantico(qid, k=5)

    n_tfidf = ndcg_at_k(r_tfidf, qid)
    n_bm25  = ndcg_at_k(r_bm25,  qid)
    n_sem   = ndcg_at_k(r_sem,   qid)

    ndcgs['tfidf'].append(n_tfidf)
    ndcgs['bm25'].append(n_bm25)
    ndcgs['sem'].append(n_sem)

    mejor = " ← semántico gana" if n_sem > max(n_tfidf, n_bm25) else ""
    print(f"{qid:<35} {n_tfidf:>8.4f} {n_bm25:>8.4f} {n_sem:>10.4f}{mejor}")

print('-' * 65)
print(f"{'nDCG medio':<35} {sum(ndcgs['tfidf'])/len(ndcgs['tfidf']):>8.4f} "
      f"{sum(ndcgs['bm25'])/len(ndcgs['bm25']):>8.4f} "
      f"{sum(ndcgs['sem'])/len(ndcgs['sem']):>10.4f}")



Consulta                              TF-IDF     BM25  Semántico
-----------------------------------------------------------------
sequia y cultivos                     1.0000   1.0000     1.0000
precio del cafe                       0.9468   0.9468     1.0000 ← semántico gana
turismo en chiapas                    0.3706   0.2937     0.8811 ← semántico gana
escasez de agua                       0.8308   0.8308     0.8428 ← semántico gana
tecnologia e innovacion en chiapas    0.2530   0.5874     0.8811 ← semántico gana
-----------------------------------------------------------------
nDCG medio                            0.6802   0.7317     0.9210


_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_

Mejoró demasiado en _"turismo en chiapas"_ y _"tecnologia e innovacion en chiapas"_ incrementó 0.5 y 0.3 approximadamente.

Esto se debe a que estos textos utilizan un vocabulario diferente al de las consultas, TF-IDF y BM25 no podían encontrar documentos coincidentes porque no tenían las mismas palabras exactas, y esto es precisamente lo que va a solucionar el semántico

## Entregables — Lab 5
- [x] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [x] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [x] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [x] `AI_USAGE.md` actualizado si usaron IA.
